In [20]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DATE = pd.Timestamp("2026-06-12")

In [6]:
#Ucitavanje svih tabela

csv_files = {
    "appearances": "appearances.csv",
    "club_games": "club_games.csv",
    "clubs": "clubs.csv",
    "competitions": "competitions.csv",
    "countries": "countries.csv",
    "game_events": "game_events.csv",
    "game_lineups": "game_lineups.csv",
    "games": "games.csv",
    "national_teams": "national_teams.csv",
    "player_valuations": "player_valuations.csv",
    "players": "players.csv",
    "transfers": "transfers.csv",
}

dfs = {}

for name, filename in csv_files.items():
    path = RAW_DIR / filename
    dfs[name] = pd.read_csv(path, low_memory=False)
    print(f"{name}: {dfs[name].shape}")

appearances: (1859048, 13)
club_games: (173380, 11)
clubs: (796, 17)
competitions: (67, 11)
countries: (118, 8)
game_events: (1238239, 11)
game_lineups: (2811747, 10)
games: (86690, 23)
national_teams: (118, 17)
player_valuations: (616377, 6)
players: (44905, 26)
transfers: (158214, 10)


In [7]:
#Pregled strukture svih tabela

summary = []

for name, df in dfs.items():
    summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary).sort_values("rows", ascending=False)
summary_df

,table,rows,columns,missing_values,duplicate_rows
6,game_lineups,2811747,10,3,0
0,appearances,1859048,13,2,0
5,game_events,1238239,11,1775624,0
9,player_valuations,616377,6,67812,0
1,club_games,173380,11,104268,0
11,transfers,158214,10,116645,0
7,games,86690,23,93055,0
10,players,44905,26,172499,0
2,clubs,796,17,1597,0
4,countries,118,8,0,0


In [8]:
#Pregled svih kolona u tabelama (nazivi)

for name, df in dfs.items():
    print(f"\n=== {name} ===")
    print(df.columns.tolist())


=== appearances ===
['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']

=== club_games ===
['game_id', 'club_id', 'own_goals', 'own_position', 'own_manager_name', 'opponent_id', 'opponent_goals', 'opponent_position', 'opponent_manager_name', 'hosting', 'is_win']

=== clubs ===
['club_id', 'club_code', 'name', 'domestic_competition_id', 'total_market_value', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_name', 'stadium_seats', 'net_transfer_record', 'coach_name', 'last_season', 'filename', 'url']

=== competitions ===
['competition_id', 'competition_code', 'name', 'sub_type', 'type', 'country_id', 'country_name', 'domestic_league_code', 'confederation', 'total_clubs', 'url']

=== countries ===
['country_id', 'country_name', 'country_code', 'confederation', 'total_clubs', 'total_playe

In [9]:
#Procenat nedostajucih vrednosti za svaku kolonu svake tabele

missing_reports = {}

for name, df in dfs.items():
    missing = (
        df.isna().mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
        .reset_index()
    )
    missing.columns = ["column", "missing_percent"]
    missing_reports[name] = missing
    
    print(f"\n=== {name} ===")
    display(missing.head(20))


=== appearances ===


,column,missing_percent
0,appearance_id,0.0
1,game_id,0.0
2,player_id,0.0
3,player_club_id,0.0
4,player_current_club_id,0.0
5,date,0.0
6,player_name,0.0
7,competition_id,0.0
8,yellow_cards,0.0
9,red_cards,0.0



=== club_games ===


,column,missing_percent
0,own_position,29.09
1,opponent_position,29.09
2,opponent_manager_name,0.97
3,own_manager_name,0.97
4,own_goals,0.00
5,game_id,0.00
6,club_id,0.00
7,opponent_goals,0.00
8,opponent_id,0.00
9,hosting,0.00



=== clubs ===


...

,column,missing_percent
0,player_club_domestic_competition_id,11.0
1,player_id,0.0
2,date,0.0
3,market_value_in_eur,0.0
4,current_club_name,0.0
5,current_club_id,0.0



=== players ===


,column,missing_percent
0,current_national_team_id,100.00
1,international_caps,66.85
2,international_goals,66.85
3,agent_name,45.88
4,contract_expiration_date,34.24
5,highest_market_value_in_eur,12.35
6,market_value_in_eur,12.35
7,foot,10.46
8,country_of_birth,10.21
9,city_of_birth,9.58



=== transfers ===


,column,missing_percent
0,market_value_in_eur,38.79
1,transfer_fee,34.94
2,transfer_date,0.00
3,player_id,0.00
4,transfer_season,0.00
5,from_club_id,0.00
6,from_club_name,0.00
7,to_club_id,0.00
8,to_club_name,0.00
9,player_name,0.00


In [10]:
#Brisanje potpuno praznih kolona

for name in dfs:
    before = dfs[name].shape[1]
    dfs[name] = dfs[name].dropna(axis=1, how="all")
    after = dfs[name].shape[1]
    print(f"{name}: removed {before - after} empty columns")

appearances: removed 0 empty columns
club_games: removed 0 empty columns
clubs: removed 1 empty columns
competitions: removed 0 empty columns
countries: removed 0 empty columns
game_events: removed 0 empty columns
game_lineups: removed 0 empty columns
games: removed 0 empty columns
national_teams: removed 1 empty columns
player_valuations: removed 0 empty columns
players: removed 1 empty columns
transfers: removed 0 empty columns


In [11]:
#Ne postoje duplirani redovi, te nema potrebe da ih izbacujemo

In [12]:
#Parsiranje datuma
date_columns = {
    "appearances": ["date"],
    "game_events": ["date"],
    "game_lineups": ["date"],
    "games": ["date"],
    "player_valuations": ["date"],
    "players": ["date_of_birth", "contract_expiration_date"],
    "transfers": ["transfer_date"],
}

for table, cols in date_columns.items():
    for col in cols:
        if col in dfs[table].columns:
            dfs[table][col] = pd.to_datetime(dfs[table][col], errors="coerce")

In [15]:
dfs["transfers"]["transfer_fee"].head(20)

0     0.0
1     0.0
2     NaN
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
Name: transfer_fee, dtype: float64

In [16]:
dfs["transfers"]["transfer_fee"] = (
    dfs["transfers"]["transfer_fee"]
    .fillna(0)
)

In [17]:
dfs["transfers"]["transfer_fee"].head(20)

0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
Name: transfer_fee, dtype: float64

In [18]:
#U tabeli transfers.csv, kolona transfer_fee je sadrzala dosta nedostajucih vrednosti. Iz tog razloga, kako ne bih izbacivao te transfere iz opticaja,
# pretpostavio sam da se radi o transferima koji su se odvili bez naknade, odnosno po ceni 0, kao i mnogi drugi transferi iz tabele

In [19]:
#Ciscenje numerickih kolona

numeric_columns = {
    "appearances": [
        "yellow_cards", "red_cards", "goals", "assists", "minutes_played"
    ],
    "club_games": [
        "own_goals", "opponent_goals", "is_win"
    ],
    "clubs": [
        "squad_size", "average_age", "foreigners_number",
        "foreigners_percentage", "national_team_players",
        "stadium_seats", "last_season"
    ],
    "competitions": [
        "country_id", "total_teams"
    ],
    "countries": [
        "country_id", "total_clubs", "total_players", "average_age"
    ],
    "game_events": [
        "minute", "club_id", "player_id", "player_in_id", "player_assist_id"
    ],
    "game_lineups": [
        "player_id", "club_id", "number", "team_captain"
    ],
    "games": [
        "season", "home_club_id", "away_club_id",
        "home_club_goals", "away_club_goals", "attendance"
    ],
    "national_teams": [
        "national_team_id", "country_id", "squad_size", "average_age",
        "foreigners_number", "foreigners_percentage",
        "total_market_value", "fifa_ranking", "last_season"
    ],
    "player_valuations": [
        "player_id", "market_value_in_eur", "current_club_id"
    ],
    "players": [
        "player_id", "last_season", "current_club_id",
        "height_in_cm", "international_caps", "international_goals",
        "current_national_team_id", "market_value_in_eur",
        "highest_market_value_in_eur"
    ],
    "transfers": [
        "player_id", "from_club_id", "to_club_id",
        "transfer_fee", "market_value_in_eur"
    ],
}

for table, cols in numeric_columns.items():
    for col in cols:
        if col in dfs[table].columns:
            dfs[table][col] = pd.to_numeric(dfs[table][col], errors="coerce")

In [21]:
#Izvlacenje starosti kao novog atributa iz datuma rodjenja. Razlog lezi u cinjenici da je jednostavnije raditi sa numerickim vrednostima.

players = dfs["players"].copy()

reference_date = REFERENCE_DATE

players["age"] = (
    (reference_date - players["date_of_birth"]).dt.days / 365.25
)

players["age"] = players["age"].round(2)

dfs["players"] = players

In [23]:
print(players[["name", "age"]].sort_values("age").head(10))
print(players[["name", "age"]].sort_values("age").tail(10))

                         name    age
44800        Gabriel Rajković  15.30
44310         Patrick Budescu  15.76
43461           Matthew Arana  15.79
44611         Beytullah Gezer  16.05
44615     Alexander Andersson  16.13
44367  Leandro Diaz Rodriguez  16.17
44826          Marko Vuckovic  16.18
44550           Eirik Granaas  16.22
44783            Oscar Avilez  16.27
44764         Zebedee Kennedy  16.30
                    name  age
29184      Ibrahim Suaib  NaN
29284      Sergiy Rusyan  NaN
31218          Kalle Bak  NaN
33449      Cihat Topatan  NaN
33858       José Salinas  NaN
35617    Luca Novodomsky  NaN
36416          Max Wendt  NaN
37034          Sem Bonte  NaN
40998  Hjalte Toftegaard  NaN
41350      Demba N'Diaye  NaN


In [24]:
players["age"].isna().sum()

np.int64(49)

In [25]:
100 * players["age"].isna().mean()

np.float64(0.10911925175370225)

In [26]:
# Vidimo da postoje igraci kojima nije upisan datum rodjenja. Medjutim, kako je njihov procenat izuzetno mali (0.1%), za sada
# cemo ih ostaviti takvima kakvi jesu, a kada se budemo pozabavili ozbiljnijim racunom i klasterovanjem, popunicemo im godine 
# medijanom preostalih godina

In [27]:
#Izbacujemo kolone koje ne predstavljaju znacajne informacije prilikom klasterovanja, kao sto su slike i drugi url-ovi

columns_to_drop = {
    "clubs": ["url", "filename"],
    "competitions": ["url"],
    "countries": ["url"],
    "games": ["url"],
    "national_teams": ["url", "team_image_url"],
    "players": ["url", "image_url"],
}

for table, cols in columns_to_drop.items():
    existing_cols = [col for col in cols if col in dfs[table].columns]
    dfs[table] = dfs[table].drop(columns=existing_cols)

In [28]:
# Cuvamo ociscene tabele

for name, df in dfs.items():
    output_path = PROCESSED_DIR / f"{name}_clean.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

Saved: ../data/processed/appearances_clean.csv
Saved: ../data/processed/club_games_clean.csv
Saved: ../data/processed/clubs_clean.csv
Saved: ../data/processed/competitions_clean.csv
Saved: ../data/processed/countries_clean.csv
Saved: ../data/processed/game_events_clean.csv
Saved: ../data/processed/game_lineups_clean.csv
Saved: ../data/processed/games_clean.csv
Saved: ../data/processed/national_teams_clean.csv
Saved: ../data/processed/player_valuations_clean.csv
Saved: ../data/processed/players_clean.csv
Saved: ../data/processed/transfers_clean.csv


In [29]:
# Na kraju, provera onoga sto smo uradili

for name, df in dfs.items():
    print(f"{name}_clean.csv: {df.shape}")

appearances_clean.csv: (1859048, 13)
club_games_clean.csv: (173380, 11)
clubs_clean.csv: (796, 14)
competitions_clean.csv: (67, 10)
countries_clean.csv: (118, 7)
game_events_clean.csv: (1238239, 11)
game_lineups_clean.csv: (2811747, 10)
games_clean.csv: (86690, 22)
national_teams_clean.csv: (118, 14)
player_valuations_clean.csv: (616377, 6)
players_clean.csv: (44905, 24)
transfers_clean.csv: (158214, 10)
